In [1]:
import pandas as pd
import os
import numpy as np
from matplotlib import pyplot as plt
import tensorflow as tf
from tensorflow.keras import optimizers
from datetime import datetime
from sklearn import metrics
from sklearn.metrics import confusion_matrix, accuracy_score

In [2]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")


In [3]:
Npontos = 60 #Tempo de dados 60
features = 6

In [4]:
dataset = pd.read_csv('normalizado.csv')

In [5]:
dataset.head()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,90426757696641,4.972757,-0.158317,6.696732,0.314944,-1.022277,-0.309962
1,1600,0,90426807196641,3.253720,-0.191835,6.107758,0.387382,-0.618541,-0.048972
2,1600,0,90426856696641,2.801216,-0.155922,5.997625,0.070999,-0.209480,-0.195978
3,1600,0,90426906196641,3.770868,-1.051354,7.731027,0.037975,0.254976,-0.156563
4,1600,0,90426955696641,4.661511,0.169689,9.684695,0.073129,0.719431,-0.001035


In [6]:
def cria_IN_OUT(data, janela, features):
    OUT_indices = np.arange(janela, len(data), janela)
    #print("imprimindo out_indices")
    #print(OUT_indices)
    OUT = data[OUT_indices]
    #print("imprimindo out")
    #print(OUT)
    IN_indices = [range(i-janela, i) for i in OUT_indices] 
    #print("imprimindo IN_INDICES")
    #print(IN_indices)  
    #print("imprimindo IN_indices flattened") 
    #print(np.array(IN_indices).flatten())
    IN = data[np.array(IN_indices).flatten()]
    #print("imprimindo IN")
    #print(IN)      
    IN = np.reshape(IN, (len(OUT), janela, features))
    IN_final = IN[:, :, :]
    return IN_final

In [7]:
parc_data_train = 0.8
featuresX = ['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']
trainX = np.empty((0, 60, len(featuresX)), dtype=float)
testX = np.empty((0, 60, len(featuresX)), dtype=float)
trainY = np.empty((0, 60, 1), dtype=float)
testY = np.empty((0, 60, 1), dtype=float)
lista_ativ = [0,3,15,2]
train1 = pd.DataFrame()


for i in range(len(lista_ativ)):
    parte = dataset.loc[dataset['atividade'] == lista_ativ[i]]
    #parte = parte.replace({lista_ativ[i]:i})
    split = int(len(parte) * parc_data_train)
    train = parte[:split]
    train1 = pd.concat([train1,train],ignore_index = True)
    print("imprimindo train shape")
    print(train.shape)
    teste = parte[split:]
    array_label_train = train["atividade"].values
    print("imprimindo label train length")
    print(len(array_label_train))
    print("imprimindo label train")
    print(array_label_train)
    array_label_test = teste["atividade"].values
    print("imprimindo array label test length")
    print(len(array_label_test))
    arraytrain = train[featuresX].values
    print("imprimindo araytrain")
    print(len(arraytrain))
    arrayteste = teste[featuresX].values
    print("imprimindo arraytest")
    print(len(arrayteste))
    transform = cria_IN_OUT(arraytrain, Npontos, features)
    transform2 = cria_IN_OUT(arrayteste, Npontos, features)
    transform3 = cria_IN_OUT(array_label_train, Npontos, 1)
    transform4 = cria_IN_OUT(array_label_test, Npontos, 1)
    trainX = np.concatenate([trainX, transform])
    testX = np.concatenate([testX, transform2])
    trainY = np.concatenate([trainY, transform3])
    testY = np.concatenate([testY, transform4])
print(trainX.shape)
print(trainY.shape)
print(testX.shape)
print(testY.shape)
print(train1.shape)

imprimindo train shape
(148919, 9)
imprimindo label train length
148919
imprimindo label train
[0 0 0 ... 0 0 0]
imprimindo array label test length
37230
imprimindo araytrain
148919
imprimindo arraytest
37230
imprimindo train shape
(154075, 9)
imprimindo label train length
154075
imprimindo label train
[3 3 3 ... 3 3 3]
imprimindo array label test length
38519
imprimindo araytrain
154075
imprimindo arraytest
38519
imprimindo train shape
(154216, 9)
imprimindo label train length
154216
imprimindo label train
[15 15 15 ... 15 15 15]
imprimindo array label test length
38555
imprimindo araytrain
154216
imprimindo arraytest
38555
imprimindo train shape
(146611, 9)
imprimindo label train length
146611
imprimindo label train
[2 2 2 ... 2 2 2]
imprimindo array label test length
36653
imprimindo araytrain
146611
imprimindo arraytest
36653
(10061, 60, 6)
(10061, 60, 1)
(2513, 60, 6)
(2513, 60, 1)
(603821, 9)


In [ ]:
#train1 = train1[train1["atividade"].isin([0,2,3,15])]

In [ ]:
train1.shape

In [ ]:
#teste de Label

parar = False
j = 0
problema = 0;
ok = 0

for i in range(0,trainY.shape[0],1):    
    for x in range(0,trainY.shape[1],1):
        #print(dataset_train.iloc[j]["atividade"])
        #print(f"imprimindo [i][x][0] - {i} - {x} ")
        #print(trainY[i][x][0])
        if train1.loc[j]["atividade"]!=trainY[i][x][0]:
            #print("problema")
            parar = True
            problema = problema + 1
            j = j+1                
        else:
            ok = ok +1
            j=j+1
               

print(problema)
print(ok)
                            

In [ ]:
#teste de Label
j = 0
for i in range(0,trainY.shape[0],1):
    teste = False
    for x in range(0,trainY.shape[1],1):
        if teste == False:
            teste = trainY[i][x][0]
        else:
            if teste != trainY[i][x][0]:
                print('errado na posição %d, %d' % (i, x))


In [ ]:
#teste de Label
j = 0
for i in range(0,trainY.shape[0],1):
    for x in range(0,trainY.shape[1],1):
        j = j+1
print(j)        


In [ ]:
print(accuracy_score(trainY[0,:],teste))

In [ ]:
# taxa de aprendizado padrão ADAM 0.001
optimizer = optimizers.Adam(clipvalue=1.0)

In [ ]:
def cria_LSTM(hidden_units, dense_units, input_shape, activation, optimizer):
  model = tf.keras.Sequential()
  model.add(tf.keras.layers.LSTM(hidden_units, input_shape=input_shape,activation=activation[0], return_sequences=True))
  #model.add(tf.keras.layers.Dense(units=64, activation=activation[0]))
  model.add(tf.keras.layers.Dropout(rate=0.5))
  #model.add(tf.keras.layers.Dense(units=32, activation=activation[0]))
  model.add(tf.keras.layers.Dense(units=dense_units, activation=activation[1]))
  model.compile(loss='sparse_categorical_crossentropy', optimizer ='adam', metrics=['accuracy'])
  return model

def reset_model():
  global model
  model = cria_LSTM(hidden_units=120, dense_units=4, input_shape=(Npontos, features), activation=['tanh', 'softmax'], optimizer=tf.keras.optimizers.Adam())

model = cria_LSTM(hidden_units=120, dense_units=4, input_shape=(Npontos, features), activation=['tanh', 'softmax'], optimizer=optimizer)

In [ ]:
historico = []
for i in range(10):
    reset_model()
    print('Rodada %d' % i)
    history = model.fit(trainX, trainY, epochs = 50, batch_size = 50, verbose = 1,  validation_data=(testX, testY))
    historico.append(history)


In [ ]:
plt.figure(figsize= (7,7))

plt.subplot(2,1,1)
plt.title("Loss - {}".format(date))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'], 'k')
plt.legend(['treino', 'teste'])

plt.subplot(2,1,2)
plt.title("acurácia")
plt.ylim(0,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'], 'k')
plt.legend(['treino', 'teste'])
plt.xlabel('epoch')

In [ ]:
nome = 'Acurácia_{}'.format(date)

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(20, 7))
axes = axes.flatten()

for i in range(10):
    ax = axes[i]
    ax.set_title(f'Acurácia - Treino {i + 1}')
    ax.plot(historico[i].history['accuracy'], label='Treino')
    ax.plot(historico[i].history['val_accuracy'], label='Validação')
    ax.set_xlabel('Épocas')
    ax.set_ylabel('Acurácia')
    ax.set_ylim(0, 1)
    ax.legend()

plt.suptitle('Acurácia {}'.format(date))
plt.savefig('Plots/%s' % arq)
plt.tight_layout()